# 04 — Outlier detection & treatment

Runs `src/preprocessing/outliers.treat_outliers`, **reconstructed** from
`outlier-treatment.ipynb`. Input `gurgaon_properties_cleaned_v2.csv` (3 803, 23),
output `gurgaon_properties_outlier_treated.csv` (3 555, 24).

## This stage is two structurally different things

**`outlier-treatment.ipynb` does not fully reproduce its own committed output.**
Three transformations in `outlier_treated.csv` have no code cell — they were
recovered by diffing input against output — and the notebook has no `to_csv`
cell either. So `outliers.py` splits cleanly in two:

1. **Derivable logic** — real, rule-based functions: `drop_duplicates`, the
   `price_per_sqft`-IQR area rescale + recompute, the threshold filters, the
   full `price_per_sqft` recompute, the `area_room_ratio` column. This is what
   would run on any fresh scrape.

2. **Historical replay patches** — hand-curated constant tables keyed to *exact
   row positions* in this one `cleaned_v2.csv`:
   - `_MANUAL_*` — the notebook's own hand-typed `df.drop(index=[...])` /
     `df.loc[idx, 'area'] = ...` edits (cells 33 / 35 / 67).
   - `_RECONSTRUCTED_*` — a 33-row drop and 70 `house` `bedRoom` corrections
     that have **no cell at all**; recovered from the input→output diff.

   **These do not generalise.** On any other input they patch the wrong rows.
   They exist only so this stage reproduces the one historical output file.

Full write-up: `PROJECT_PLAN.md` Section 9, and the "second instance" note under
Section 1's untraced-step paragraph. This notebook makes it visible.

In [1]:
import sys, logging
from pathlib import Path

REPO_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)

from src.preprocessing import outliers as O
from src.preprocessing.outliers import treat_outliers

INTERIM = REPO_ROOT / "data" / "interim"


## 1. The derivable logic

In notebook order:

- **`drop_duplicates()`** on the raw load (3 803 → 3 677) — no `subset`, no log
  in the original; logged here.
- **`price_per_sqft` IQR outliers** — for those rows, an `area < 1000` is
  assumed to be square yards and multiplied by 9, then `price_per_sqft` is
  recomputed as `price / area`.
- **threshold filters** — `price_per_sqft ≤ 50 000`, `area < 100 000`,
  `bedRoom ≤ 10`. Rows with a null `price` / `area` fall out here as a
  `NaN`-comparison side effect (not a `dropna`).
- **`price_per_sqft` recomputed** for every row.
- **`area_room_ratio` = `area / bedRoom`** — the 24th column (no cell creates
  it; see part 2).

In [2]:
cv2 = pd.read_csv(INTERIM / "gurgaon_properties_cleaned_v2.csv")
print("raw load        :", cv2.shape)
print("after dedupe    :", cv2.drop_duplicates().shape)
print("\nnulls in the input (columns with any):")
print(cv2.isnull().sum()[cv2.isnull().sum() > 0])


raw load        : (3803, 23)
after dedupe    : (3677, 23)

nulls in the input (columns with any):
society                   1
price                    18
price_per_sqft           18
area                     18
floorNum                 19
facing                 1105
super_built_up_area    1888
built_up_area          2070
carpet_area            1859
dtype: int64


In [3]:
treated = treat_outliers(cv2)
print("\ninput  :", cv2.shape)
print("output :", treated.shape, " (24th column: area_room_ratio)")


drop_duplicates: 3803 -> 3677 row(s)


price_per_sqft IQR outliers rescaled: 354 row(s)


price_per_sqft/area threshold filters: 3677 -> 3642 row(s)


bedRoom <= 10 filter: 3633 -> 3588 row(s)


reconstructed garbage-row drop: 33 row(s)


reconstructed house bedRoom corrections: 70 row(s)



input  : (3803, 23)
output : (3555, 24)  (24th column: area_room_ratio)


In [4]:
treated.head()


,property_type,society,sector,price,price_per_sqft,area,areaWithType,bedRoom,bathroom,balcony,...,built_up_area,carpet_area,study room,servant room,store room,pooja room,others,furnishing_type,luxury_score,area_room_ratio
0,flat,signature global park 4,sector 36,0.82,7586.0,1081.0,Super Built up area 1081(100.43 sq.m.)Carpet a...,3.0,2.0,2,...,NaN,650.0,0.0,0.0,0.0,0.0,0.0,0.0,8.0,360.333333
1,flat,smart world gems,sector 89,0.95,8597.0,1105.0,Carpet area: 1103 (102.47 sq.m.),2.0,2.0,2,...,NaN,1103.0,1.0,1.0,0.0,0.0,0.0,0.0,38.0,552.500000
2,flat,breez global hill view,sohna road,0.32,5470.0,585.0,Built Up area: 1000 (92.9 sq.m.)Carpet area: 5...,2.0,2.0,1,...,1000.0,585.0,0.0,0.0,0.0,0.0,0.0,0.0,49.0,292.500000
3,flat,bestech park view sanskruti,sector 92,1.60,8020.0,1995.0,Super Built up area 1995(185.34 sq.m.)Built Up...,3.0,4.0,3+,...,1615.0,1476.0,0.0,1.0,0.0,0.0,1.0,1.0,174.0,665.000000
4,flat,suncity avenue,sector 102,0.48,9023.0,532.0,Super Built up area 632(58.71 sq.m.)Carpet are...,2.0,2.0,1,...,NaN,532.0,0.0,0.0,1.0,0.0,0.0,0.0,159.0,266.000000


In [5]:
# exact-match check against the committed file — values AND dtypes
expected = pd.read_csv(INTERIM / "gurgaon_properties_outlier_treated.csv")
assert list(treated.columns) == list(expected.columns)
assert treated.shape == expected.shape
assert list(treated.dtypes.astype(str)) == list(expected.dtypes.astype(str)), "dtype mismatch"
total = match = 0
for col in expected.columns:
    a, b = treated[col], expected[col]
    m = (np.isclose(a.astype(float), b.astype(float), equal_nan=True)
         if b.dtype.kind in "fi" else a.astype(str) == b.astype(str))
    total += len(m); match += int(m.sum())
print(f"cell match vs committed file: {match}/{total} = {100 * match / total:.4f}%")
print("dtypes identical:", list(treated.dtypes.astype(str)) == list(expected.dtypes.astype(str)))


cell match vs committed file: 85320/85320 = 100.0000%
dtypes identical: True


## 2. The historical replay patches

Every key below is a **positional row index into `cleaned_v2.csv`** (labels
survive `drop_duplicates`; kept until the final `reset_index`). None of this
generalises to a different input.

| constant | source | entries | generalises? |
|---|---|---|---|
| `_MANUAL_AREA_DROP` | notebook cell 33 — `df.drop(index=[...])` | 9 | no |
| `_MANUAL_AREA_FIXES` | notebook cell 35 — `df.loc[idx, 'area'] = value` | 8 | no |
| `_MANUAL_CARPET_FIXES` | notebook cell 67 — `df.loc[2131, 'carpet_area'] = 1812` | 1 | no |
| `_RECONSTRUCTED_ROW_DROP` | **no cell** — recovered from the input→output diff | 33 | no |
| `_RECONSTRUCTED_HOUSE_BEDROOM` | **no cell** — recovered from the diff | 70 | no |
| `_UPDATE_UPCASTS_TO_FLOAT` | reproduces a pandas < 2.0 `DataFrame.update` dtype quirk | 9 cols | n/a (dtype only) |

In [6]:
print("_MANUAL_AREA_DROP    :", O._MANUAL_AREA_DROP)
print("_MANUAL_AREA_FIXES   :", O._MANUAL_AREA_FIXES)
print("_MANUAL_CARPET_FIXES :", O._MANUAL_CARPET_FIXES)
print("_UPDATE_UPCASTS_TO_FLOAT :", O._UPDATE_UPCASTS_TO_FLOAT)
print()
print("_RECONSTRUCTED_ROW_DROP (%d):" % len(O._RECONSTRUCTED_ROW_DROP))
print(O._RECONSTRUCTED_ROW_DROP)
print()
print("_RECONSTRUCTED_HOUSE_BEDROOM (%d entries):" % len(O._RECONSTRUCTED_HOUSE_BEDROOM))
print(O._RECONSTRUCTED_HOUSE_BEDROOM)


_MANUAL_AREA_DROP    : [818, 1796, 1123, 2, 2356, 115, 3649, 2503, 1471]
_MANUAL_AREA_FIXES   : {48: 1035, 300: 7250, 2666: 5800, 1358: 2660, 3195: 2850, 2131: 1812, 3088: 2160, 3444: 1175}
_MANUAL_CARPET_FIXES : {2131: 1812}
_UPDATE_UPCASTS_TO_FLOAT : ['bedRoom', 'bathroom', 'study room', 'servant room', 'store room', 'pooja room', 'others', 'furnishing_type', 'luxury_score']

_RECONSTRUCTED_ROW_DROP (33):
[37, 93, 229, 247, 387, 751, 753, 1106, 1206, 1429, 1562, 1580, 1696, 1737, 1747, 1773, 1936, 1939, 1953, 1997, 2047, 2300, 2360, 2721, 2784, 2806, 3148, 3246, 3268, 3306, 3329, 3633, 3774]

_RECONSTRUCTED_HOUSE_BEDROOM (70 entries):
{9: 3, 15: 2, 48: 3, 74: 2, 99: 3, 140: 2, 186: 2, 255: 2, 293: 2, 343: 3, 393: 3, 530: 2, 540: 3, 565: 2, 668: 3, 837: 4, 848: 2, 852: 3, 880: 3, 886: 1, 935: 2, 1033: 2, 1049: 3, 1087: 3, 1167: 2, 1187: 3, 1213: 2, 1224: 2, 1384: 2, 1407: 3, 1480: 4, 1500: 2, 1509: 2, 1524: 2, 1532: 1, 1537: 2, 1627: 2, 1798: 3, 1851: 1, 1951: 2, 2004: 2, 2063: 3, 217

### What `_RECONSTRUCTED_ROW_DROP` actually removes

33 rows the notebook author dropped by hand with **no cell** — implausibly
small `area` for the `bedRoom` count (a 50 sqft "5-bedroom", a ₹1.85 Cr / 60
sqft listing). They loosely track "low `area / bedRoom`" but no threshold
reproduces exactly these 33 (tried `area/bedRoom` cutoffs, `price_per_sqft`,
combinations), so they can only be replayed by index. Shown here as they appear
in `cleaned_v2` (before the IQR square-yard rescale):

In [7]:
deduped = cv2.drop_duplicates()   # labels preserved, as treat_outliers uses them
drop_rows = deduped.loc[O._RECONSTRUCTED_ROW_DROP,
                        ["sector", "price", "area", "bedRoom"]].copy()
drop_rows["area_per_bedroom"] = (drop_rows["area"] / drop_rows["bedRoom"]).round(1)
drop_rows.sort_values("area_per_bedroom").head(15)


,sector,price,area,bedRoom,area_per_bedroom
2047,sector 43,1.85,60.0,8,7.5
229,sector 12,0.60,50.0,5,10.0
1696,sector 86,0.42,50.0,5,10.0
3774,sector 28,0.45,50.0,5,10.0
3148,sector 6,0.85,67.0,5,13.4
2784,sector 28,0.75,360.0,7,51.4
2300,sector 5,0.40,450.0,7,64.3
2360,sector 105,0.60,540.0,8,67.5
1562,sector 6,0.50,360.0,5,72.0
751,sector 17,0.32,145.0,2,72.5


### `_RECONSTRUCTED_HOUSE_BEDROOM` — 70 `house` rows corrected downward

`bedRoom` reduced from 4–10 to 1–4, **only** for `property_type == 'house'`.
No formula fits. It is also **not** a join to `houses.csv` /
`independent-house.csv` — those carry the *original* inflated counts (checked
separately: raw `bedRoom` matches the pre-correction value 56/57 times, the
corrected value 0/57).

In [8]:
bed = pd.DataFrame({
    "sector": deduped.loc[list(O._RECONSTRUCTED_HOUSE_BEDROOM), "sector"].values,
    "property_type": deduped.loc[list(O._RECONSTRUCTED_HOUSE_BEDROOM), "property_type"].values,
    "bedRoom (was)": deduped.loc[list(O._RECONSTRUCTED_HOUSE_BEDROOM), "bedRoom"].values,
    "bedRoom (corrected)": list(O._RECONSTRUCTED_HOUSE_BEDROOM.values()),
})
print("all rows are property_type == 'house':", bool((bed["property_type"] == "house").all()))
print("correction is always downward       :",
      bool((bed["bedRoom (corrected)"] < bed["bedRoom (was)"]).all()))
bed.head(15)


all rows are property_type == 'house': True
correction is always downward       : True


,sector,property_type,bedRoom (was),bedRoom (corrected)
0,sector 105,house,6,3
1,sector 12,house,4,2
2,sector 13,house,10,3
3,sector 25,house,9,2
4,sector 4,house,6,3
5,sector 28,house,4,2
6,sector 13,house,10,2
7,sector 49,house,6,2
8,sector 57,house,8,2
9,sector 38,house,9,3


## What this means for reuse

Run `treat_outliers` on a **new scrape** and the derivable logic (part 1) does
the right thing, but the replay patches (part 2) silently mis-target rows by
position. To reuse this stage properly, either regenerate the
`_RECONSTRUCTED_*` / `_MANUAL_*` tables from a fresh input→output diff, or —
better — reconstruct the missing notebook cells as real rules. Until then this
module reproduces *one* historical file and nothing more.